# Diarka QAOA Portfolio — Notebook 02: QUBO and Ising Encoding

**Week 2, Session 2.** Goal: encode the binary portfolio problem as a QUBO and then as an Ising Hamiltonian, by hand, and verify that the ground state of the Hamiltonian is exactly the brute-force optimum from Session 1.

In Session 1 we let `qiskit-finance` perform this encoding silently inside `PortfolioOptimization`. Here we do it explicitly. Three reasons:

1. **Understanding.** QAOA solves an Ising Hamiltonian. If we don't know what Hamiltonian we're handing it, we can't reason about its behaviour, its parameter landscape, or its failure modes.
2. **Documentation.** The encoding becomes a reviewable artefact (`src/encoding.py`) we can point to in the blog post and LinkedIn carousel, rather than a black-box dependency.
3. **Flexibility.** Later we may want non-standard penalty terms or alternative formulations (e.g. soft target-return constraints). A hand-rolled encoding makes those modifications a one-line change rather than a fight with a library.

The path is:

$$
\underbrace{\text{cardinality-constrained mean-variance}}_{\text{Session 1}}
\quad\Longrightarrow\quad
\underbrace{\text{QUBO}}_{\S\,1{-}3}
\quad\Longrightarrow\quad
\underbrace{\text{Ising Hamiltonian}}_{\S\,4{-}6}
\quad\Longrightarrow\quad
\underbrace{\text{QAOA-ready operator}}_{\text{Session 3}}
$$

## 0. Setup and load Session 1 outputs

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data import PortfolioStats
from src.encoding import (
    QUBOFormulation,
    build_qubo,
    choose_penalty,
    qubo_to_ising,
    hamiltonian_eigendecomposition,
    bitstring_to_selection,
)

DATA_PROCESSED = ROOT / "data" / "processed"

# Load the annualised statistics persisted by Session 1.
stats = PortfolioStats.from_npz(DATA_PROCESSED / "portfolio_stats.npz")

# Load the brute-force classical solution from Session 1 — this is the
# ground-truth target the QUBO and Ising encodings must reproduce.
classical = np.load(DATA_PROCESSED / "classical_solution.npz", allow_pickle=False)
X_CLASSICAL = classical["selection"]
RISK_FACTOR = float(classical["risk_factor"])
BUDGET      = int(classical["budget"])
CLASSICAL_OBJ = float(classical["objective"])

print(f"Universe:    {stats.tickers}")
print(f"n assets:    {stats.n_assets}")
print(f"Budget B:    {BUDGET}")
print(f"Risk q:      {RISK_FACTOR}")
print(f"Classical x*: {X_CLASSICAL.astype(int).tolist()}")
print(f"Classical f*: {CLASSICAL_OBJ:.6f}  (objective: q·xᵀΣx − μᵀx)")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 1. From constrained problem to QUBO

The cardinality-constrained mean-variance problem is:

$$
\min_{x \in \{0,1\}^n} \quad q \cdot x^{\!\top} \Sigma\, x \;-\; \mu^{\!\top} x
\qquad \text{s.t.} \quad \sum_{i=1}^n x_i = B
$$

QUBOs (and Ising models, and QAOA) cannot handle equality constraints directly — they minimise an *unconstrained* quadratic over binary variables. The standard trick is to absorb the constraint as a quadratic **penalty term**:

$$
\min_{x \in \{0,1\}^n} \quad
\underbrace{q\, x^{\!\top}\Sigma x - \mu^{\!\top} x}_{\text{original objective}}
\;+\;
\underbrace{P\left(\sum_i x_i - B\right)^2}_{\text{penalty: 0 iff } \sum x_i = B}
$$

Any infeasible $x$ pays a penalty of at least $P \cdot 1 = P$ (off by one asset). If $P$ is large enough to outweigh any possible improvement an infeasible $x$ could offer in the original objective, the unconstrained problem has the same optimum as the constrained one.

Expanding the square gives

$$
\left(\sum_i x_i - B\right)^2 = \sum_{i,j} x_i x_j \;-\; 2B\sum_i x_i \;+\; B^2
$$

so the penalty contributes:

- a **quadratic** term $P \sum_{i,j} x_i x_j$ — $P$ on every pair (including $i=j$),
- a **linear** term $-2BP \sum_i x_i$,
- a **constant** $PB^2$ (irrelevant for the argmin but tracked for the absolute value).

The full QUBO is $f(x) = x^{\!\top} Q\, x + \text{offset}$ with:

| element | value |
|---|---|
| $Q_{ii}$ (diagonal) | $q\,\Sigma_{ii} + P - \mu_i - 2BP$ |
| $Q_{ij}$ (off-diagonal) | $q\,\Sigma_{ij} + P$ |
| offset | $PB^2$ |

We use the binary identity $x_i^2 = x_i$ to fold $-\mu$ and $-2BP$ onto the diagonal of $Q$, giving a clean homogeneous-quadratic form. This is exactly what `build_qubo()` in `src/encoding.py` constructs.

### Picking the penalty $P$

A common safe heuristic:

$$P \;=\; 2 \cdot \left(\,|\mu|_\infty + q \cdot |\Sigma|_\infty\,\right)$$

This makes the penalty for being off-budget by one asset bigger than the largest possible improvement an infeasible bitstring could win in the unconstrained objective. `choose_penalty()` implements this.

In [ ]:
penalty = choose_penalty(stats.mu, stats.sigma, RISK_FACTOR)
print(f"Chosen penalty P = {penalty:.4f}")
print(f"  |μ|_∞       = {np.max(np.abs(stats.mu)):.4f}")
print(f"  q·|Σ|_∞     = {RISK_FACTOR * np.max(np.abs(stats.sigma)):.4f}")
print(f"  P should comfortably exceed both. ✓")

## 2. Build the QUBO

In [ ]:
qubo = build_qubo(
    stats.mu, stats.sigma,
    budget=BUDGET,
    risk_factor=RISK_FACTOR,
    penalty=penalty,
    tickers=stats.tickers,
)

print(f"QUBO summary")
print(f"  n        = {qubo.n}")
print(f"  budget   = {qubo.budget}")
print(f"  q        = {qubo.risk_factor}")
print(f"  P        = {qubo.penalty:.4f}")
print(f"  offset   = {qubo.offset:.4f}  (= P·B² = {qubo.penalty:.4f} · {qubo.budget}²)")
print()
print("Q matrix (annualised, including penalty):")
for i, row in enumerate(qubo.Q):
    label = qubo.tickers[i] if qubo.tickers else f"x{i}"
    print(f"  {label:8s}  " + "  ".join(f"{v:+.4f}" for v in row))

## 3. Sanity check: does the QUBO recover the brute-force optimum?

The QUBO objective adds the penalty term to the original objective. On any *feasible* $x$ (one satisfying $\sum x_i = B$) the penalty is zero, so

$$f(x) = q\,x^{\!\top}\Sigma x - \mu^{\!\top}x \quad \text{whenever } \sum x_i = B.$$

That means: on feasible bitstrings the QUBO and Session 1's brute-force objective are identical, so they must share the same argmin. Anywhere off the feasible set, the QUBO is *strictly larger* by at least $P$.

Below we enumerate all $2^n = 2^8 = 256$ bitstrings, evaluate the QUBO on each, and:

1. Confirm the argmin matches the brute-force selection from Session 1.
2. Confirm every feasible bitstring's QUBO value equals its classical objective.
3. Visualise the landscape — feasible vs infeasible — to make the penalty's role visible.

In [12]:
bitstrings, qubo_values = qubo.evaluate_all()
cardinalities = bitstrings.sum(axis=1).astype(int)

# QUBO minimum.
k_star = int(np.argmin(qubo_values))
x_star = bitstrings[k_star]
f_star = qubo_values[k_star]

print(f"QUBO minimum")
print(f"  x*     = {x_star.astype(int).tolist()}")
print(f"  f(x*)  = {f_star:.6f}")
print(f"  ∑x*    = {int(x_star.sum())}  (budget was {BUDGET})")
print()
print(f"Classical (Session 1) selection")
print(f"  x*     = {X_CLASSICAL.astype(int).tolist()}")
print(f"  obj    = {CLASSICAL_OBJ:.6f}")

assert np.allclose(x_star, X_CLASSICAL), "QUBO optimum disagrees with Session 1!"
assert abs(f_star - CLASSICAL_OBJ) < 1e-9, "QUBO value at optimum disagrees with classical objective!"
print()
print("✓ QUBO argmin matches the brute-force selection from Session 1.")
print("✓ QUBO value at the optimum matches the classical objective (penalty contributes 0 on feasible x).")

QUBO minimum
  x*     = [1, 1, 1, 1, 0, 0, 0, 0]
  f(x*)  = -0.267284
  ∑x*    = 4  (budget was 4)

Classical (Session 1) selection
  x*     = [1, 1, 1, 1, 0, 0, 0, 0]
  obj    = -0.267284

✓ QUBO argmin matches the brute-force selection from Session 1.
✓ QUBO value at the optimum matches the classical objective (penalty contributes 0 on feasible x).


### Visualising the landscape

Below: every bitstring's QUBO value, separated by cardinality. The feasible set (cardinality $= B = 4$) is highlighted; everything else carries a penalty.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

colors = []
for c in cardinalities:
    if c == BUDGET:
        colors.append("#1f77b4")           # feasible — blue
    else:
        colors.append("#cccccc")           # infeasible — light grey

ax.scatter(cardinalities, qubo_values, c=colors, s=20, alpha=0.8)
ax.axvline(BUDGET, color="#1f77b4", linestyle="--", alpha=0.5,
           label=f"feasible: ∑x = {BUDGET}")

# Mark the optimum.
ax.scatter([cardinalities[k_star]], [qubo_values[k_star]],
           c="#d62728", s=120, zorder=5, marker="*", label="QUBO minimum")

ax.set_xlabel("Cardinality  ∑ xᵢ")
ax.set_ylabel("QUBO value  f(x)")
ax.set_title("QUBO landscape over all 2⁸ = 256 bitstrings")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Tighter verification: on every feasible bitstring, QUBO value equals the
# original classical objective q·xᵀΣx − μᵀx (penalty contributes zero).
feasible_mask = (cardinalities == BUDGET)
n_feasible = int(feasible_mask.sum())
print(f"Feasible bitstrings: {n_feasible}  (= C({stats.n_assets}, {BUDGET}) = {n_feasible})")

classical_objs = np.array([
    RISK_FACTOR * (x @ stats.sigma @ x) - stats.mu @ x
    for x in bitstrings[feasible_mask]
])
qubo_objs_feasible = qubo_values[feasible_mask]

max_gap = float(np.max(np.abs(qubo_objs_feasible - classical_objs)))
print(f"Max |QUBO − classical| on feasible set: {max_gap:.2e}")
assert max_gap < 1e-9, "QUBO and classical disagree on the feasible set!"
print("✓ QUBO and classical objective agree on every feasible bitstring.")

## 4. From QUBO to Ising Hamiltonian

QAOA's quantum part minimises the expectation value of an Ising Hamiltonian on $n$ qubits. To get there from a QUBO, substitute

$$x_i \;=\; \frac{1 - z_i}{2}, \qquad z_i \in \{-1, +1\}$$

This maps each $\{0,1\}$ variable to a $\{-1,+1\}$ spin. After expanding $f(x) = x^{\!\top} Q x + \text{offset}$ with $Q$ symmetric:

$$
f \;=\; \text{const}_{\text{ising}} \;+\; \sum_{i} h_i\, z_i \;+\; \sum_{i<j} J_{ij}\, z_i z_j
$$

with

$$
\begin{aligned}
h_i &= -\tfrac{1}{2} \sum_j Q_{ij} \\
J_{ij} &= \tfrac{1}{2}\, Q_{ij} \quad (i < j) \\
\text{const}_{\text{ising}} &= \tfrac{1}{4} \sum_{i,j} Q_{ij} \;+\; \tfrac{1}{4} \operatorname{tr}(Q) \;+\; \text{offset}_{\text{QUBO}}
\end{aligned}
$$

Promoting each $z_i$ to a Pauli-$Z$ operator on qubit $i$ gives a Hermitian operator

$$
\hat H \;=\; \sum_i h_i\, Z_i \;+\; \sum_{i<j} J_{ij}\, Z_i Z_j
$$

This is what we hand to QAOA in Session 3.

### Mapping to selections

In the standard quantum-computing convention $|0\rangle$ is the $+1$ eigenstate of $Z$ and $|1\rangle$ is the $-1$ eigenstate. So:

- $z_i = +1 \;\Leftrightarrow\; |0\rangle_i \;\Leftrightarrow\; x_i = (1-1)/2 = 0$
- $z_i = -1 \;\Leftrightarrow\; |1\rangle_i \;\Leftrightarrow\; x_i = (1+1)/2 = 1$

A measurement bitstring $b_1 b_2 \ldots b_n$ is therefore *literally* the selection vector $x$. The lowest-energy computational-basis state of $\hat H$ is the QUBO (and classical) optimum.

In [ ]:
H, ising_offset = qubo_to_ising(qubo)

print(f"Ising Hamiltonian")
print(f"  qubits      = {H.num_qubits}")
print(f"  Pauli terms = {len(H)}")
print(f"  ising_offset = {ising_offset:.6f}")
print()
print(f"Recall: ⟨H⟩ + ising_offset = f(x)  (the QUBO objective)")
print()
print("First few Pauli terms (label, coefficient):")
for label, coeff in list(zip(H.paulis.to_labels(), H.coeffs))[:10]:
    print(f"  {label}   {coeff.real:+.6f}")
print(f"  ... ({len(H) - 10} more)")

## 5. Diagonalise the Hamiltonian and find its ground state

Because $\hat H$ contains only $I$ and $Z$ Paulis, it is **diagonal in the computational basis**. Its eigenvalues are just the diagonal entries of the $2^n \times 2^n$ matrix; no actual diagonalisation in the linear-algebra sense is needed — we read them off directly.

For $n = 8$ this gives 256 eigenvalues, one per computational-basis state. The smallest of these is the ground-state energy; the basis state that achieves it is the ground state.

We then verify three things:

1. The ground-state bitstring (interpreted as a selection $x$) matches the Session 1 classical solution.
2. The ground-state energy plus `ising_offset` equals the QUBO optimum from §3.
3. The full set of 256 sorted Ising energies, once offset back, matches the full set of 256 sorted QUBO values from §3.

In [ ]:
energies_sorted, bitstrings_sorted = hamiltonian_eigendecomposition(H)

ground_energy   = energies_sorted[0]
ground_bits_arr = bitstrings_sorted[0]
ground_bitstring = "".join(str(b) for b in reversed(ground_bits_arr))  # Qiskit convention

print(f"Ground state of H")
print(f"  energy                    = {ground_energy:.6f}")
print(f"  bitstring (Qiskit)        = {ground_bitstring}")
print(f"  selection (q0..q{stats.n_assets-1}) = {ground_bits_arr.tolist()}")
print(f"  ⟨H⟩ + ising_offset        = {ground_energy + ising_offset:.6f}")
print()
print(f"For comparison (Session 1)")
print(f"  classical selection       = {X_CLASSICAL.astype(int).tolist()}")
print(f"  classical objective       = {CLASSICAL_OBJ:.6f}")

assert np.array_equal(ground_bits_arr, X_CLASSICAL.astype(int)), (
    "Ising ground state disagrees with classical solution!"
)
assert abs((ground_energy + ising_offset) - CLASSICAL_OBJ) < 1e-9, (
    "Ising energy + offset disagrees with classical objective!"
)
print()
print("✓ Ising ground state = QUBO optimum = Session 1 classical solution.")

In [ ]:
# Full spectrum check: do the 256 Ising energies (offset back) match the
# 256 QUBO values? They should agree as multi-sets.
ising_values_as_qubo = np.sort(energies_sorted + ising_offset)
qubo_values_sorted   = np.sort(qubo_values)

max_gap = float(np.max(np.abs(ising_values_as_qubo - qubo_values_sorted)))
print(f"Max disagreement across full spectrum: {max_gap:.2e}")
assert max_gap < 1e-9, "Ising and QUBO spectra do not agree!"
print("✓ Every Ising eigenvalue equals (after offset) a QUBO value, and vice versa.")

## 6. The energy landscape QAOA will be navigating

QAOA's job in Session 3 will be to prepare a quantum state whose expectation value $\langle H \rangle$ is as close as possible to the ground-state energy. The shape of the spectrum tells us what 'as close as possible' looks like in practice — in particular, the **spectral gap** (energy difference between ground state and first excited state) determines how distinctive the optimum is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram of Ising energies, with the ground state highlighted.
ax = axes[0]
ax.hist(energies_sorted, bins=40, color="#cccccc", edgecolor="white")
ax.axvline(ground_energy, color="#d62728", linestyle="--",
           label=f"ground state E = {ground_energy:.4f}")
ax.set_xlabel("Ising energy")
ax.set_ylabel("Count")
ax.set_title("Distribution of all 2⁸ = 256 Ising eigenvalues")
ax.legend()

# Right: sorted spectrum, with the spectral gap annotated.
ax = axes[1]
ax.plot(range(len(energies_sorted)), energies_sorted, "-", color="#1f77b4", alpha=0.6)
ax.scatter([0], [energies_sorted[0]], color="#d62728", s=80, zorder=5,
           label=f"ground (E={energies_sorted[0]:.4f})")
ax.scatter([1], [energies_sorted[1]], color="#ff7f0e", s=80, zorder=5,
           label=f"1st excited (E={energies_sorted[1]:.4f})")
ax.set_xlabel("State index (sorted)")
ax.set_ylabel("Ising energy")
ax.set_title(f"Sorted spectrum — gap = {energies_sorted[1] - energies_sorted[0]:.4f}")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Spectral gap (E1 − E0) = {energies_sorted[1] - energies_sorted[0]:.6f}")
print(f"This is the energy QAOA must beat to confidently identify the optimum.")

## 7. Persist outputs for Session 3

Session 3 (first QAOA on a simulator) needs:

- The Pauli decomposition of $\hat H$ (so it can construct the cost layer).
- The Ising offset (so it can convert ⟨H⟩ back to portfolio objective values).
- The ground-state bitstring and energy (so it can compute the approximation ratio).
- The full sorted spectrum (so it can plot QAOA's sampling distribution against the true energies).

We persist these as `data/processed/hamiltonian.npz`.

In [ ]:
pauli_labels = np.array(H.paulis.to_labels())
pauli_coeffs = np.array([c.real for c in H.coeffs])

np.savez(
    DATA_PROCESSED / "hamiltonian.npz",
    pauli_labels=pauli_labels,
    pauli_coeffs=pauli_coeffs,
    ising_offset=ising_offset,
    ground_energy=ground_energy,
    ground_bitstring=ground_bitstring,
    ground_selection=ground_bits_arr,
    sorted_energies=energies_sorted,
    sorted_bitstrings=bitstrings_sorted,
    Q_matrix=qubo.Q,
    qubo_offset=qubo.offset,
    penalty=qubo.penalty,
    n_qubits=H.num_qubits,
    tickers=np.array(stats.tickers),
)

print("Saved:")
print(f"  {(DATA_PROCESSED / 'hamiltonian.npz').relative_to(ROOT)}")
print(f"    pauli_labels:    {pauli_labels.shape}  (Pauli strings)")
print(f"    pauli_coeffs:    {pauli_coeffs.shape}")
print(f"    ising_offset:    scalar  ({ising_offset:.6f})")
print(f"    ground_energy:   scalar  ({ground_energy:.6f})")
print(f"    ground_bitstring: '{ground_bitstring}'")
print(f"    sorted_energies: {energies_sorted.shape}")

## Session 2 — wrap-up

What we have now:

- A documented, hand-rolled QUBO encoding (`src/encoding.py`) that reproduces qiskit-finance's encoding *and* the brute-force selection from Session 1, with every step traceable.
- An Ising Hamiltonian as a `SparsePauliOp` whose lowest-energy computational-basis state is exactly the optimal portfolio.
- A persisted artefact (`data/processed/hamiltonian.npz`) carrying everything Session 3 needs to construct a QAOA ansatz and measure approximation quality against ground truth.

**Next session (Session 3).** Build the QAOA circuit for this Hamiltonian:

1. Construct the cost layer $e^{-i\gamma \hat H}$ as a parameterised circuit.
2. Construct the mixer layer $e^{-i\beta X}$ on each qubit.
3. Optimise $(\gamma, \beta)$ classically (COBYLA) against $\langle H \rangle$ on the Aer simulator.
4. Sample the optimised circuit, plot the bitstring distribution against the true Ising spectrum, and compute the approximation ratio.
5. Repeat for $p = 1, 2, 3$ and observe how the distribution sharpens around the ground state.